In [2]:
import ads
import os
ads.config.token = os.environ['ADS_TOKEN']  # this is supposed to happen automatically but it doesn't

3 styles, long (all authors with mccully bold), short (all et al for collaboration, first 3 for major contribution articles), medium (first 3 for all publications, \[\]then my name then brackets for collaboration)

In [3]:
def format_journal(journal):
    if journal.lower() == 'a&a':
        journal='aap'
    return f'\\{journal.lower()}'

def format_title(title):
    title = title.replace('&', '\\&')
    title = title.replace('%', '\\%')
    title = title.replace(r'\&amp;', r'\&')
    title = title.replace('\u223c', r'\texttildelow')
    title = title.replace('<SUP>\u2605</SUP>', r'$^{\star}$')
    title = title.replace('\u03b1', r'\textalpha')
    
    return title

def format_author(author, last_name):

    if len(author.split(', ')) == 1:
        return f'{{{author}}}\n'
    
    if 'Jr.' in author:
        author = author.replace(', Jr.', '')
        junior = True
    else:
        junior = False
    author_last_name, author_rest_name = author.split(', ')

    author_rest_name = author_rest_name.replace(' -', '-')

    if len(author_rest_name.split()) > 1:
        # There is a middle initial
        middle_initial = f"~{author_rest_name[author_rest_name.index(' ') + 1:]}"
        first_name = author_rest_name.split()[0]
    else:
        middle_initial = ''
        first_name = author_rest_name
        
    if '-' in first_name:
        # Don't forget to strip the period off the end
        first_initial = '-'.join([f'{name[0].upper()}.' for name in first_name.split('-')])[:-1]
    else:
        first_initial = author_rest_name[0]
    
    if last_name.lower() in author.lower():
        formatted_author = f'{{\\textbf{{{author_last_name}}}}}, \\textbf{{{first_initial}.{middle_initial}}}'
    else:
        formatted_author = f'{{{author_last_name}}}, {first_initial}.{middle_initial}'
    if junior:
        formatted_author += ', Jr.'
    return formatted_author

def format_authors(paper, last_name, fmt='short', medium_length=4):
    # use these as the author lists
    override_authors = {'2017A&A...607A.115I': '{IceCube Collaboration} {et~al.}',
                        '2017Natur.551...85A': '{LIGO/Virgo Collaboration}\n {et~al.}',
                        '2017ApJ...848L..12A': '{LIGO/Virgo Collaboration}\n {et~al.}',
                        '2018AJ....156..123A': '{Astropy Collaboration}'}
    if paper.bibcode in override_authors:
        return f'{override_authors[paper.bibcode]}\n'
    if fmt=='short':
        authors = f'{{{format_author(paper.author[0], last_name)}}}, {{et~al.}}'
    elif fmt == 'medium':
        authors = ', '.join([format_author(author, last_name) for author in paper.author[:medium_length]])
        if any([last_name.lower() in author.lower() for author in paper.author[:medium_length]]):
            if len(paper.author) <= medium_length:
                # replace the last comma with an ampersand
                split_authors = authors.rsplit(sep=', ', maxsplit=2)
                authors = f'{split_authors[0]}, \\& {split_authors[1]}, {split_authors[0]}'
            else:
                authors += f', {{[{len(paper.author) - medium_length}~Authors]}}'
        else:
            # Find me in the author list
            my_index = [last_name.lower() in author.lower() for author in paper.author].index(True)
            if my_index == medium_length:
                authors += f', {format_author(paper.author[medium_length], last_name)}, {{[{len(paper.author) - medium_length - 1}~Authors]}}'
            elif my_index == len(paper.author) - 1:
                authors += f', {{[{len(paper.author) - medium_length - 1}~Authors]}}, {format_author(paper.author[-1], last_name)}'
            else:
                authors += f', {{[{my_index - medium_length}~Authors]}}, {format_author(paper.author[my_index], last_name)}, {{[{len(paper.author) - my_index - 1}~Authors]}}'
    else:
        authors = ', '.join([f'{format_author(author, last_name)}' for author in paper.author])
        if len(paper.author) > 1:
            split_authors = authors.rsplit(sep=', ', maxsplit=2)
            authors = f'{split_authors[0]}, \\& {split_authors[1]}, {split_authors[2]}'
    
    authors += '\n'
    return authors

def format_bibitem(paper, last_name, fmt='short'):
    bib_entry = f'\\bibitem{{{paper.bibcode}}}\n'
    bib_entry += format_authors(paper, last_name, fmt=fmt)
    bib_entry += '\\newblock\n'
    bib_entry += f"\\newblock {paper.year}, {{\\em ``{{{format_title(paper.title[0])}}}''}}, "
    if 'arXiv' in paper.pub:
        arxiv_number = paper.page[0].split(':')[1]
        bib_entry += f"\\href{{https://arxiv.org/abs/{arxiv_number}}}{{{{\\em arXiv }}: {arxiv_number}}}"
    else:
        if paper.doi is not None:
            bib_entry += f"\\href{{https://doi.org/{paper.doi[0]}}}{{{{\\em {format_journal(paper.bibstem[0])}}}}}, "
        else:
            bib_entry += f"{{\\em {format_journal(paper.bibstem[0])}}}, "
        if paper.volume is None:
            bib_entry += f'\\href{{https://ui.adsabs.harvard.edu/abs/{paper.bibcode}}}{{in press}}'
        else:
            bib_entry += f'\\href{{https://ui.adsabs.harvard.edu/abs/{paper.bibcode}}}{{{paper.volume}, {paper.page[0]}}}'
    bib_entry += '.\n\n'
    return bib_entry

def make_bbl_file(output_filename, papers, last_name, fmt='short'):
    with open(output_filename, 'w') as f:
        f.write(f'\\begin{{thebibliography}}{{{len(papers)}}}\n\n')
        for paper in papers:
            f.write(format_bibitem(paper, last_name, fmt=fmt))
        f.write(r'\end{thebibliography}')

In [ ]:
header= r"""\documentclass[colorlinks,linkcolor=true,linkcolor=blue,urlcolor=blue]{moderncv}
\moderncvstyle{classic}
\moderncvcolor{black}

\newcommand\mnras{MNRAS}
\newcommand\aj{AJ}
\newcommand\apj{ApJ}
\newcommand\aap{A\&A}
\newcommand\apjs{ApJS}
\newcommand\nat{Nature}
\newcommand\apjl{ApJL}
\newcommand\sci{Science}
\newcommand\natur{Nature}
\newcommand\natas{Nature Astronomy}
\newcommand\pasa{PASA}
\newcommand\pasp{PASP}
\newcommand\aas{AAS}
\newcommand\spie{SPIE}
\newcommand\an{Astronomische Nachrichten}
\newcommand\psj{PSJ}
\newcommand\univ{Universe}

\renewcommand*{\namefont}{\fontsize{18}{22}\mdseries\upshape}


"""

name_section = """
\\firstname{{{first_name}}}
\\familyname{{{last_name}}}

"""

body = r"""
\usepackage[letterpaper, portrait, margin=1in]{geometry}
%----------------------------------------------------------------------------------------
%   NAME AND CONTACT INFORMATION SECTION
%----------------------------------------------------------------------------------------

%% bibliography with multiple entries
\usepackage{multibib}
\newcites{firstauthor,majorcontrib,collab,proceedings}{{First Author Journal Articles},{Major Contribution Journal Articles},{Collaboration Journal Articles},{Proceedings}}
\usepackage[numbers]{natbib}
\usepackage{textgreek}

%%%------------------------------------------------------------------
%%% Reverse numbering of bibliography
\usepackage{etaremune,etoolbox}
\makeatletter
\AtBeginDocument{%%% natbib redefines the environment there
\renewenvironment{thebibliography}[1]
 {\bibsection\parindent\z@\bibpreamble\bibfont
  \settowidth{\dimen0}{#1.}%
  \setlength{\dimen2}{\dimen0}%
  \addtolength{\dimen2}{\labelsep}
  \begin{etaremune}[labelwidth=\dimen0,leftmargin=\dimen2]
  \ifNAT@openbib
    \renewcommand\newblock{\par}%
  \else
    \renewcommand\newblock{\hskip.11em \@plus .33em \@minus .07em}%
  \fi
  \sloppy
  \clubpenalty4000
  \widowpenalty4000
  \sfcode`\.\@m
  \let\NAT@bibitem@first@sw\@firstoftwo
  \let\citeN\cite\let\shortcite\cite\let\citeasnoun\cite}
 {\bibitem@fin\bibpostamble
  \def\@noitemerr{\PackageWarning{natbib}{Empty `thebibliography' environment}}%
  \end{etaremune}
  \bibcleanup}
}%%% end of \AtBeginDocument

%%% patch \@lbibitem to use only \item (for etaremune)
\patchcmd{\@lbibitem}{\item[\hfil\NAT@anchor{#2}{\NAT@num}]}{\item}{}{}
\makeatother
\begin{document}
\title{Publications}
\makecvtitle
\vspace*{-10mm}
%%------------------------------------------------------------------
\rule{\textwidth}{1pt}

\nocitefirstauthor{*}
\bibliographyfirstauthor{firstauthor}

\nocitemajorcontrib{*}
\bibliographymajorcontrib{majorcontrib}

\nocitecollab{*}
\bibliographycollab{collab}
%
\rule{\textwidth}{1pt}
\nociteproceedings{*}
\bibliographyproceedings{proceedings}
\rule{\textwidth}{1pt}

"""
gcn_section = """
\\section{{{circulars} GCNs, ATels, CBETs, and TNS Reports on Supernovae and Gravitational Wave Follow up}}\n
"""
footer = r"""%----------------------------------------------------------------------------------------
\end{document}
"""

In [ ]:
def write_publication_list(first_name, last_name, fmt='medium'):
    name = f'{last_name}, {first_name}'
    journal_query = f'author:"{name}" (property:REFEREED OR bibstem:arXiv)'            # papers
    
    papers = ads.SearchQuery(q=journal_query, sort='date', rows=1000,
                         fl=['author', 'first_author', 'year', 'title', 'pub',
                             'volume', 'page', 'doi', 'bibcode', 'bibstem'])
    papers = [paper for paper in papers]

    # Filter out Errata and PhD Thesis
    papers = [paper for paper in papers 
              if 'phd' not in paper.bibcode.lower() and 
              'errat' not in paper.title[0].lower() and 
              'corrig' not in paper.title[0].lower()] 
    first_author_papers = [paper for paper in papers if last_name.lower() in paper.author[0].lower()]
    major_contribution_papers = [paper for paper in papers if any([last_name.lower() in paper.author[i].lower() for i in range(1,3)])]
    collaboration_papers = [paper for paper in papers if not any([last_name.lower() in paper.author[i].lower() for i in range(3)])]
    
    make_bbl_file('firstauthor.bbl', first_author_papers, last_name, fmt=fmt)
    make_bbl_file('majorcontrib.bbl', major_contribution_papers, last_name, fmt=fmt)
    make_bbl_file('collab.bbl', collaboration_papers, last_name, fmt=fmt)
    
    proceedings_query = f'author:"{name}" (bibstem:SPIE OR bibstem:AAS)'
    
    proceedings = ads.SearchQuery(q=proceedings_query, sort='date', rows=1000,
                                  fl=['author', 'first_author', 'year', 'title', 'pub',
                                      'volume', 'page', 'doi', 'bibcode', 'bibstem'])
    proceedings = [proceeding for proceeding in proceedings]
    make_bbl_file('proceedings.bbl', proceedings, last_name, fmt=fmt)

    circular_query = f'author: "{name}" bibstem:(ATel OR TNSCR OR CBET OR GCN)'  # all circulars 
    
    circulars = ads.SearchQuery(q=circular_query, sort='date', rows=1000,
                                fl=['author', 'first_author', 'year', 'title', 'pub',
                                    'volume', 'page', 'doi', 'bibcode', 'bibstem'])
    circulars = [circular for circular in circulars]
    with open('publications.tex', 'w') as f:
        f.write(header)
        f.write(name_section.format(first_name=first_name, last_name=last_name))
        f.write(body)
        f.write(gcn_section.format(circulars=len(circulars)))
        f.write(footer)
    
    for i in range(3):
        os.system('pdflatex publications')

In [ ]:
write_publication_list('Curtis', 'McCully', fmt='long')

.def))))
(/usr/local/texlive/2022/texmf-dist/tex/latex/moderncv/moderncviconstikz.sty
(/usr/local/texlive/2022/texmf-dist/tex/latex/pgf/frontendlayer/tikz.sty
(/usr/local/texlive/2022/texmf-dist/tex/latex/pgf/basiclayer/pgf.sty
(/usr/local/texlive/2022/texmf-dist/tex/latex/pgf/utilities/pgfrcs.sty
(/usr/local/texlive/2022/texmf-dist/tex/generic/pgf/utilities/pgfutil-common.te
x
(/usr/local/texlive/2022/texmf-dist/tex/generic/pgf/utilities/pgfutil-common-li
sts.tex))
(/usr/local/texlive/2022/texmf-dist/tex/generic/pgf/utilities/pgfutil-latex.def
) (/usr/local/texlive/2022/texmf-dist/tex/generic/pgf/utilities/pgfrcs.code.tex
(/usr/local/texlive/2022/texmf-dist/tex/generic/pgf/pgf.revision.tex)))
(/usr/local/texlive/2022/texmf-dist/tex/latex/pgf/basiclayer/pgfcore.sty
(/usr/local/texlive/2022/texmf-dist/tex/latex/pgf/systemlayer/pgfsys.sty
(/usr/local/texlive/2022/texmf-dist/tex/generic/pgf/systemlayer/pgfsys.code.tex
(/usr/local/texlive/2022/texmf-dist/tex/generic/pgf/utilities/pgfkeys.c

(/usr/local/texlive/2022/texmf-dist/tex/context/base/mkii/supp-pdf.mkii
[Loading MPS to PDF converter (version 2006.09.02).]
) (/usr/local/texlive/2022/texmf-dist/tex/latex/epstopdf-pkg/epstopdf-base.sty
(/usr/local/texlive/2022/texmf-dist/tex/latex/latexconfig/epstopdf-sys.cfg))
(/usr/local/texlive/2022/texmf-dist/tex/latex/microtype/mt-cmr.cfg)
*geometry* driver: auto-detecting
*geometry* detected driver: pdftex
(/usr/local/texlive/2022/texmf-dist/tex/latex/hyperref/nameref.sty
(/usr/local/texlive/2022/texmf-dist/tex/latex/refcount/refcount.sty)
(/usr/local/texlive/2022/texmf-dist/tex/generic/gettitlestring/gettitlestring.s
ty)) (./publications.out) (./publications.out)

LaTeX Font Warning: Font shape `OT1/cmr/m/n' in size <18> not available
(Font)              size <17.28> substituted on input line 73.


Underfull \hbox (badness 10000) in paragraph at lines 73--73

(./firstauthor.bbl) (./majorcontrib.bbl [1{/usr/local/texlive/2022/texmf-var/fo
nts/map/pdftex/updmap/pdftex.map}]) (./

(/usr/local/texlive/2022/texmf-dist/tex/latex/geometry/geometry.sty
(/usr/local/texlive/2022/texmf-dist/tex/generic/iftex/ifvtex.sty))
(/usr/local/texlive/2022/texmf-dist/tex/latex/multibib/multibib.sty)
(./firstauthor.aux) (./majorcontrib.aux) (./collab.aux) (./proceedings.aux)
(/usr/local/texlive/2022/texmf-dist/tex/latex/natbib/natbib.sty)
(/usr/local/texlive/2022/texmf-dist/tex/latex/textgreek/textgreek.sty
(/usr/local/texlive/2022/texmf-dist/tex/latex/greek-fontenc/lgrenc.def
(/usr/local/texlive/2022/texmf-dist/tex/latex/greek-inputenc/lgrenc.dfu)
(/usr/local/texlive/2022/texmf-dist/tex/latex/greek-fontenc/greek-fontenc.def))
) (/usr/local/texlive/2022/texmf-dist/tex/latex/etaremune/etaremune.sty
(/usr/local/texlive/2022/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/local/texlive/2022/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/local/texlive/2022/texmf-dist/tex/generic/xkeyval/xkvutils.tex))))
(/usr/local/texlive/2022/texmf-dist/tex/latex/hyperref/hyperref.sty
(/usr/local/te